# Getting Started

This notebook showcases basic functionality of the code base.

Here, we load the metadata, an example dataset, and run inference using a pre-trained model. 

We also show how to visualize the joint angle predictions using a hand mesh (requires the UmeTrack package -- see README.md).

In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
from pathlib import Path
DATA_DOWNLOAD_DIR = Path.home()

## Download Dataset Metadata

In [ ]:
!cd {DATA_DOWNLOAD_DIR} && curl https://fb-ctrl-oss.s3.amazonaws.com/emg2pose/emg2pose_metadata.csv -o emg2pose_metadata.csv

In [3]:
import pandas as pd

metadata_df = pd.read_csv(DATA_DOWNLOAD_DIR / "emg2pose_metadata.csv")
metadata_df.head(5)

,session,user,stage,start,end,side,filename,moving_hand,held_out_user,held_out_stage,split,generalization
0,2022-04-07-1649318400-8125c-cv-emg-pose-train@2,29ddab35d7,ThumbsUpDownThumbRotationsCWCCWP,1.649400e+09,1.649400e+09,left,2022-04-07-1649318400-8125c-cv-emg-pose-train@...,both,True,False,val,user
1,2022-04-07-1649318400-8125c-cv-emg-pose-train@2,29ddab35d7,ThumbsUpDownThumbRotationsCWCCWP,1.649400e+09,1.649400e+09,right,2022-04-07-1649318400-8125c-cv-emg-pose-train@...,both,True,False,val,user
2,2022-04-07-1649318400-8125c-cv-emg-pose-train@2,29ddab35d7,HandClawGraspFlicks,1.649401e+09,1.649401e+09,left,2022-04-07-1649318400-8125c-cv-emg-pose-train@...,both,True,False,val,user
3,2022-04-07-1649318400-8125c-cv-emg-pose-train@2,29ddab35d7,HandClawGraspFlicks,1.649401e+09,1.649401e+09,right,2022-04-07-1649318400-8125c-cv-emg-pose-train@...,both,True,False,val,user
4,2022-04-07-1649318400-8125c-cv-emg-pose-train@2,29ddab35d7,ShakaVulcanPeace,1.649401e+09,1.649401e+09,left,2022-04-07-1649318400-8125c-cv-emg-pose-train@...,both,True,True,val,user_stage


## Download a Smaller (~600 MiB) Version of the Dataset

In [ ]:
!cd {DATA_DOWNLOAD_DIR} && curl "https://fb-ctrl-oss.s3.amazonaws.com/emg2pose/emg2pose_dataset_mini.tar" -o emg2pose_dataset_mini.tar

# Unpack the tar to ~/emg2pose_dataset_mini
!tar -xvf emg2pose_dataset_mini.tar

In [5]:
import glob
import os

sessions = sorted(glob.glob((r"c:\Users\Leo\Documents\EMG\Data\emg2pose_dataset_mini/*.hdf5")))
sessions

['c:\\Users\\Leo\\Documents\\EMG\\Data\\emg2pose_dataset_mini\\2022-12-06-1670313600-e3096-cv-emg-pose-train@2-recording-10_left.hdf5',
 'c:\\Users\\Leo\\Documents\\EMG\\Data\\emg2pose_dataset_mini\\2022-12-06-1670313600-e3096-cv-emg-pose-train@2-recording-10_right.hdf5',
 'c:\\Users\\Leo\\Documents\\EMG\\Data\\emg2pose_dataset_mini\\2022-12-06-1670313600-e3096-cv-emg-pose-train@2-recording-11_left.hdf5',
 'c:\\Users\\Leo\\Documents\\EMG\\Data\\emg2pose_dataset_mini\\2022-12-06-1670313600-e3096-cv-emg-pose-train@2-recording-11_right.hdf5',
 'c:\\Users\\Leo\\Documents\\EMG\\Data\\emg2pose_dataset_mini\\2022-12-06-1670313600-e3096-cv-emg-pose-train@2-recording-12_left.hdf5',
 'c:\\Users\\Leo\\Documents\\EMG\\Data\\emg2pose_dataset_mini\\2022-12-06-1670313600-e3096-cv-emg-pose-train@2-recording-12_right.hdf5',
 'c:\\Users\\Leo\\Documents\\EMG\\Data\\emg2pose_dataset_mini\\2022-12-06-1670313600-e3096-cv-emg-pose-train@2-recording-13_left.hdf5',
 'c:\\Users\\Leo\\Documents\\EMG\\Data\\emg2p

## Let's Look at a Dataset

In [6]:
from emg2pose.data import Emg2PoseSessionData

session = sessions[15]
data = Emg2PoseSessionData(hdf5_path=session)

In [7]:
print(data.fields)
print()

print(f"{'emg shape: ':<20} {data['emg'].shape}")
print(f"{'joint_angles shape: ':<20} {data['joint_angles'].shape}")
print(f"{'time shape: ':<20} {data['time'].shape}")

dict_keys(['time', 'joint_angles', 'emg'])

emg shape:           (119116, 16)
joint_angles shape:  (119116, 20)
time shape:          (119116,)


In [8]:
metadata_df[metadata_df["filename"] == data.metadata["filename"]]

,session,user,stage,start,end,side,filename,moving_hand,held_out_user,held_out_stage,split,generalization
25238,2022-12-06-1670313600-e3096-cv-emg-pose-train@2,d387095792,CountingUpDownFaceSideAway,1.670313e+09,1.670313e+09,right,2022-12-06-1670313600-e3096-cv-emg-pose-train@...,both,True,True,val,user_stage


## Let's Load a Checkpoint and Generate some Predictions

In [ ]:
!cd {DATA_DOWNLOAD_DIR} \
&& curl "https://fb-ctrl-oss.s3.amazonaws.com/emg2pose/emg2pose_model_checkpoints.tar.gz" -o emg2pose_model_checkpoints.tar.gz && \
tar -xvzf emg2pose_model_checkpoints.tar.gz

In [9]:
from emg2pose.utils import generate_hydra_config_from_overrides

config = generate_hydra_config_from_overrides(
    overrides=[
        "experiment=tracking_vemg2pose",
        f"checkpoint={DATA_DOWNLOAD_DIR}/emg2pose_model_checkpoints/regression_vemg2pose.ckpt"
    ]
)

In [10]:
from emg2pose.lightning import Emg2PoseModule

module = Emg2PoseModule.load_from_checkpoint(
    config.checkpoint,
    network=config.network,
    optimizer=config.optimizer,
    lr_scheduler=config.lr_scheduler,
)

c:\Users\Leo\anaconda3\envs\emg2pose\lib\site-packages\lightning_fabric\__init__.py:41: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
Lightning automatically upgraded your loaded checkpoint from v1.8.6 to v2.2.2. To apply the upgrade to your files permanently, run `python -m pytorch_lightning.utilities.upgrade_checkpoint C:\Users\Leo\emg2pose_model_checkpoints\regression_vemg2pose.ckpt`


In [11]:
session = data
start_idx = 0
stop_idx = 10_000

In [18]:
import torch
device = next(module.parameters()).device
session_window = session[start_idx:stop_idx]

# no_ik_failure is not a field so we slice separately
no_ik_failure_window = session.no_ik_failure[start_idx:stop_idx]

batch = {
    "emg": torch.Tensor([session_window["emg"].T]),  # BCT
    "joint_angles": torch.Tensor([session_window["joint_angles"].T]),  # BCT
    "no_ik_failure": torch.Tensor([no_ik_failure_window]),  # BT
}
batch = {
    key: value.to(device)
    for key, value in batch.items()
}

preds, joint_angles, no_ik_failure = module.forward(batch)

# Algorithms that use the initial state for ground truth will do poorly
# when the first joint angles are missing!
if (joint_angles[:, 0] == 0).all():
    print(
        "Warning! Ground truth not available at first time step!"
    )

# BCT --> TC (as numpy)
preds = preds[0].cpu().T.detach().numpy()
joint_angles = joint_angles[0].cpu().T.detach().numpy()

In [19]:
preds.shape

(8210, 20)

In [20]:
joint_angles.shape

(8210, 20)